### FASE 3: División de Datos 

| Vamos a extraer caracteristicas por cada registro

Importamos librerias

In [20]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
tqdm.pandas()
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pyswarms as ps
from pyswarms.utils.functions import single_obj as fx
import seaborn as sns
import matplotlib.pyplot as plt




In [21]:
df = pd.read_csv('completo_clusters_con_features.csv')

df_model = df.copy()

le = LabelEncoder()
df_model['Direccion'] = le.fit_transform(df_model['Direccion'])


features = [
    'Total_Vehiculos', 'Tiempo_Medio_s', 'Ocupacion_Espacial_%', 
    'Hora_Minutos', 'Dia_Semana', 'Direccion', 'cluster', 
    'Es_Hora_Pico', 'Total_Vehiculos_lag1', 'Ocupacion_lag1', 
    'Media_Movil_3ciclos', 'Tendencia_Vehiculos', 'Saturacion_Actual',
    'Periodo_Dia', 'Cluster_Hora_Pico'  
]


entreno_data = df_model[df_model['Dia_Semana'].isin([1, 2,5])]
val_data   = df_model[df_model['Dia_Semana'] == 3]
prueba_data  = df_model[df_model['Dia_Semana'] == 4]


X_entreno, y_entreno = entreno_data[features], entreno_data['Tiempo_Optimo_Target']
X_val, y_val     = val_data[features], val_data['Tiempo_Optimo_Target']
X_prueba, y_prueba   = prueba_data[features], prueba_data['Tiempo_Optimo_Target']

df_model  = pd.concat([entreno_data, val_data, prueba_data])


### FASE 5: Optimización con PSO

In [ ]:

def pso_rf_optimization(X_train, y_train, X_val, y_val):
    """
    Usa PSO para optimizar hiperparámetros de Random Forest
    """
    
    def fitness_function(particles):
        """
        Función objetivo para PSO
        particles: matriz donde cada fila es una partícula (configuración)
        """
        scores = []
        
        for particle in particles:
            n_estimators = int(particle[0])
            max_depth = int(particle[1]) if particle[1] > 0 else None
            min_samples_split = int(particle[2])
            min_samples_leaf = int(particle[3])
            max_features_idx = int(particle[4])
            
            max_features_options = ['sqrt', 'log2', None]
            max_features = max_features_options[max_features_idx]
            
            rf = RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                random_state=42,
                n_jobs=-1
            )
            
            try:
                rf.fit(X_train, y_train)
                y_pred = rf.predict(X_val)
                
                mse = mean_squared_error(y_val, y_pred)
                scores.append(mse)
            except:
                
                scores.append(1e10)
        
        return np.array(scores)
    
    lb = [50, 0, 2, 1, 0]      # Lower bounds
    ub = [300, 30, 20, 10, 2]  # Upper bounds
    bounds = (lb, ub)
    
    options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}  
    optimizer = ps.single.GlobalBestPSO(
        n_particles=20,      # Número de partículas
        dimensions=5,        # Dimensiones (5 hiperparámetros)
        options=options,
        bounds=bounds
    )
    
    print("Iniciando optimización PSO...")
    best_cost, best_pos = optimizer.optimize(fitness_function, iters=20)
    
    best_params = {
        'n_estimators': int(best_pos[0]),
        'max_depth': int(best_pos[1]) if best_pos[1] > 0 else None,
        'min_samples_split': int(best_pos[2]),
        'min_samples_leaf': int(best_pos[3]),
        'max_features': ['sqrt', 'log2', None][int(best_pos[4])]
    }
    
    return best_params, best_cost

best_params, best_cost = pso_rf_optimization(X_entreno, y_entreno, X_val, y_val)

print("\n=== MEJORES HIPERPARÁMETROS ENCONTRADOS POR PSO ===")
print(best_params)
print(f"Mejor MSE: {best_cost:.3f}")

2026-01-13 20:10:08,799 - pyswarms.single.global_best - INFO - Optimize for 20 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


Iniciando optimización PSO...


pyswarms.single.global_best: 100%|██████████|20/20, best_cost=0.44 
2026-01-13 20:10:45,832 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.43952795780310394, best pos: [131.01206926  13.05539357   2.73900668   1.93498746   1.87791017]



=== MEJORES HIPERPARÁMETROS ENCONTRADOS POR PSO ===
{'n_estimators': 131, 'max_depth': 13, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Mejor MSE: 0.440


### FASE 6: Modelo Random Forest Optimizado

In [23]:
df_model = df_model.sort_values(by=['Dia_Semana', 'Hora_Minutos']).reset_index(drop=True)

X = df_model[features]
y = df_model['Tiempo_Optimo_Target']

tscv = TimeSeriesSplit(n_splits=5)

print(f"Iniciando Validación Cruzada con Expansión (Total muestras: {len(X)})")

mae_scores = []
rmse_scores = []
r2_scores = []

for i, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]
    
    
    rf_fold = RandomForestRegressor(
        **best_params, # Tus parámetros del PSO
        random_state=42,
        n_jobs=-1
    )
    rf_fold.fit(X_train_fold, y_train_fold)
    
    y_pred_fold = rf_fold.predict(X_test_fold)
    
    mae = mean_absolute_error(y_test_fold, y_pred_fold)
    rmse = np.sqrt(mean_squared_error(y_test_fold, y_pred_fold))
    r2 = r2_score(y_test_fold, y_pred_fold)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)
    
    print(f"Iteración {i+1}: Train tam={len(train_index)} | Test tam={len(test_index)} -> MAE: {mae:.3f} | R²: {r2:.4f}")



Iniciando Validación Cruzada con Expansión (Total muestras: 3801)
Iteración 1: Train tam=636 | Test tam=633 -> MAE: 0.768 | R²: 0.9773
Iteración 2: Train tam=1269 | Test tam=633 -> MAE: 0.603 | R²: 0.9862
Iteración 3: Train tam=1902 | Test tam=633 -> MAE: 0.498 | R²: 0.9899
Iteración 4: Train tam=2535 | Test tam=633 -> MAE: 0.528 | R²: 0.9898
Iteración 5: Train tam=3168 | Test tam=633 -> MAE: 0.473 | R²: 0.9891


### FASE 7: Evaluación Final 


In [24]:
print("\n=== RESULTADOS PROMEDIO (VALIDACIÓN ROBUSTA) ===")
print(f"MAE Promedio:  {np.mean(mae_scores):.3f}")
print(f"RMSE Promedio: {np.mean(rmse_scores):.3f}")
print(f"R² Promedio:   {np.mean(r2_scores):.4f}")

rf_final = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_final.fit(X, y)
print("\n✅ Modelo final entrenado con el 100% de la historia disponible.")



=== RESULTADOS PROMEDIO (VALIDACIÓN ROBUSTA) ===
MAE Promedio:  0.574
RMSE Promedio: 0.852
R² Promedio:   0.9865

✅ Modelo final entrenado con el 100% de la historia disponible.


### FASE 8: Prediccion de tiempos verdes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def comparar_vs_tiempo_fijo(test_data, y_test_pred, fijo_verde=30, amarillo=3):
    """
    Compara el modelo dinámico (predicción) vs un ciclo fijo.
    fijo_verde: 30s (El tiempo que realmente fluyen los autos)
    amarillo: 3s (Tiempo muerto/seguridad)
    """
    df = test_data.copy()
    df['Verde_Modelo'] = y_test_pred
    df['Verde_Fijo'] = fijo_verde
    
    # Diferencia: Positivo = Modelo pide más tiempo (había cola). Negativo = Modelo ahorra tiempo.
    df['Diferencia'] = df['Verde_Modelo'] - df['Verde_Fijo']
    
    print(f"\n{'='*70}")
    print(f"COMPARATIVA: MODELO DINÁMICO vs FIJO ({fijo_verde}s verde + {amarillo}s amarillo)")
    print(f"{'='*70}\n")
    
    resultados = {}
    
    for direccion in sorted(df['Direccion'].unique()):
        # Filtrar datos por dirección
        datos = df[df['Direccion'] == direccion]
        
        # 1. Cálculos de Tiempos
        promedio_modelo = datos['Verde_Modelo'].mean()
        
        # 2. Métricas de Impacto
        # Casos donde el fijo (30s) quedaba corto y se formaba cola
        casos_saturados = datos[datos['Verde_Modelo'] > fijo_verde]
        cola_evitada_prom = casos_saturados['Diferencia'].mean() if not casos_saturados.empty else 0
        freq_saturacion = (len(casos_saturados) / len(datos)) * 100
        
        # Casos donde el fijo (30s) sobraba y se desperdiciaba tiempo
        casos_holgados = datos[datos['Verde_Modelo'] < fijo_verde]
        tiempo_ahorrado_prom = abs(casos_holgados['Diferencia'].mean()) if not casos_holgados.empty else 0
        freq_ahorro = (len(casos_holgados) / len(datos)) * 100
        
        # Guardar para retorno
        resultados[f'Dir_{direccion}'] = round(promedio_modelo)
        
        print(f"🚦 DIRECCIÓN {direccion + 1}")
        print(f"   • Promedio Modelo: {promedio_modelo:.1f}s (vs {fijo_verde}s fijo)")
        print(f"   • 📉 Ahorro (Eficiencia): En el {freq_ahorro:.1f}% de ciclos, ahorramos ~{tiempo_ahorrado_prom:.1f}s cada uno.")
        print(f"   • 📈 Congestión (Eficacia): En el {freq_saturacion:.1f}% de ciclos, evitamos colas añadiendo ~{cola_evitada_prom:.1f}s.")
        print("-" * 40)


    return resultados



COMPARATIVA: MODELO DINÁMICO vs FIJO (30s verde + 3s amarillo)

🚦 DIRECCIÓN 1
   • Promedio Modelo: 25.5s (vs 30s fijo)
   • 📉 Ahorro (Eficiencia): En el 61.1% de ciclos, ahorramos ~9.9s cada uno.
   • 📈 Congestión (Eficacia): En el 38.9% de ciclos, evitamos colas añadiendo ~4.0s.
----------------------------------------
🚦 DIRECCIÓN 2
   • Promedio Modelo: 26.0s (vs 30s fijo)
   • 📉 Ahorro (Eficiencia): En el 56.3% de ciclos, ahorramos ~9.8s cada uno.
   • 📈 Congestión (Eficacia): En el 43.7% de ciclos, evitamos colas añadiendo ~3.4s.
----------------------------------------
🚦 DIRECCIÓN 3
   • Promedio Modelo: 31.0s (vs 30s fijo)
   • 📉 Ahorro (Eficiencia): En el 40.3% de ciclos, ahorramos ~8.4s cada uno.
   • 📈 Congestión (Eficacia): En el 59.7% de ciclos, evitamos colas añadiendo ~7.4s.
----------------------------------------
🚦 DIRECCIÓN 4
   • Promedio Modelo: 27.4s (vs 30s fijo)
   • 📉 Ahorro (Eficiencia): En el 44.7% de ciclos, ahorramos ~9.6s cada uno.
   • 📈 Congestión (Eficac